# Train YOLO on Dataset-3 (Google Colab with GPU)

### Quick Instructions:
1. Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.
2. Run each cell in order.
3. Upload Dataset-3_yolo.zip (45 MB) when prompted in Cell 3.

In [ ]:
# Cell 1: Check GPU
!nvidia-smi

In [ ]:
# Cell 2: Install Ultralytics YOLO
!pip install -q ultralytics

import torch
print(f"CUDA is available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 3: Extract Dataset-3_yolo.zip and configure data.yaml
import os
from google.colab import files

if not os.path.exists("Dataset-3_yolo.zip"):
    print("Please upload Dataset-3_yolo.zip from your computer...")
    uploaded = files.upload()

# Linux native unzip
!unzip -q -o Dataset-3_yolo.zip -d .

# Ensure exact Colab path in data.yaml
yaml_content = '''path: /content/Dataset-3_yolo
train: images/train
val: images/val
test: images/test

nc: 1
names: ['pothole']
'''

with open('/content/Dataset-3_yolo/data.yaml', 'w') as f:
    f.write(yaml_content.strip())

print("Extraction & data.yaml configuration complete:")
!cat /content/Dataset-3_yolo/data.yaml


In [ ]:
# Cell 4: Train YOLOv8s
from ultralytics import YOLO

# Load pretrained YOLOv8s weights
model = YOLO("yolov8s.pt")

# Train for 50 epochs (takes ~5-7 minutes on Colab T4 GPU)
results = model.train(
    data="/content/Dataset-3_yolo/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    project="runs/colab_dataset3",
    name="exp_yolov8s",
    seed=42,
    plots=True
)

In [ ]:
# Cell 5: Evaluate on Validation and Test Splits
best_weights = f"{model.trainer.save_dir}/weights/best.pt"
best_model = YOLO(best_weights)

print("=" * 50)
print("VALIDATION SET EVALUATION:")
print("=" * 50)
val_res = best_model.val(data="/content/Dataset-3_yolo/data.yaml", split="val")
print(f"Precision:   {val_res.box.p.mean():.4f}")
print(f"Recall:      {val_res.box.r.mean():.4f}")
print(f"mAP@0.5:     {val_res.box.map50:.4f}")
print(f"mAP@0.5:0.95:{val_res.box.map:.4f}")

print("\n" + "=" * 50)
print("TEST SET EVALUATION:")
print("=" * 50)
test_res = best_model.val(data="/content/Dataset-3_yolo/data.yaml", split="test")
print(f"Precision:   {test_res.box.p.mean():.4f}")
print(f"Recall:      {test_res.box.r.mean():.4f}")
print(f"mAP@0.5:     {test_res.box.map50:.4f}")
print(f"mAP@0.5:0.95:{test_res.box.map:.4f}")

In [ ]:
# Cell 6: Download Training Plots & Weights as a ZIP
import shutil
from google.colab import files

shutil.make_archive("dataset3_results", "zip", "runs/colab_dataset3")
print("Downloading results zip (plots, confusion matrix, weights)...")
files.download("dataset3_results.zip")